In [ ]:
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings 
warnings.filterwarnings('ignore')
plt.rcParams['font.family']= 'NanumGothic' 
plt.rcParams['axes.unicode_minus']=False

In [ ]:
df = pd.read_csv("df_concat_BA1.csv")

In [ ]:
df

In [ ]:
df.info(), df.head()

In [ ]:
# 기부여부 컬럼 값 분포 확인
df['기부여부'].value_counts(dropna=False)

In [ ]:
# 각 컬럼별 결측치 개수 확인
missing_counts = df.isnull().sum()
missing_counts[missing_counts > 0].sort_values(ascending=False)
## 71840 rows

In [ ]:
# 결측치가 71840개는 삭제
# 전체 행과 동일한 결측치를 가진 컬럼만 선택하여 삭제
cols_to_drop = missing_counts[missing_counts == df.shape[0]].index.tolist()

# 컬럼 삭제
df_cleaned = df.drop(columns=cols_to_drop)

# 삭제 후 남은 컬럼 수 확인(기존: 71840 rows × 257 columns)
df_cleaned.shape

In [ ]:
missing_counts_remaining = df_cleaned.isnull().sum()
missing_all = missing_counts_remaining[missing_counts_remaining > 0]

# Pandas 옵션을 바꿔 전체를 출력
pd.set_option('display.max_rows', None)

missing_all.sort_values(ascending=False)
71840/2

In [ ]:
# 각 행(row)별 결측치 개수 확인
missing_per_row = df_cleaned.isnull().sum(axis=1)
missing_per_row.value_counts().sort_index()
#(결측치 개수)    (그런 행이 몇 개인지)

In [ ]:
## 미래 기부의향: 기부예측에 그대로 쓰면 답안지 역할 가능

In [ ]:
missing_counts_remaining = df_cleaned.isnull().sum()
missing_all = missing_counts_remaining[missing_counts_remaining > 0]

# Pandas 옵션을 바꿔 전체를 출력
pd.set_option('display.max_rows', None)

missing_all.sort_values(ascending=False)

In [ ]:
71840/2

In [ ]:
# 데이터 기본 정보
info = pd.DataFrame({
    'dtype': df.dtypes,
    'missing': df.isnull().sum(),
    'unique': df.nunique()
}).sort_values(by='missing', ascending=False)

info

In [ ]:
import pandas as pd
import numpy as np

def custom_missing_handler(df):
    # ===============================
    # 고결측 변수 삭제 (60% 이상)
    # ===============================
    total_rows = len(df)
    high_missing_cols = df.columns[df.isnull().sum() / total_rows > 0.6]
    df = df.drop(columns=high_missing_cols, errors='ignore')
    print(f"삭제한 고결측 컬럼 수: {len(high_missing_cols)}")

    # ===============================
    #  연속형 변수 (시간수/횟수/권수) → 0 대체
    # ===============================
    continous_patterns = ['시간수', '횟수', '권수', '일수']
    continous_cols = [col for col in df.columns if any(p in col for p in continous_patterns)]
    df[continous_cols] = df[continous_cols].fillna(0)
    print(f"연속형 컬럼 수: {len(continous_cols)} → 결측 0 대체")

    # ===============================
    # Yes/No 변수 (참석여부, 납부여부, 소유여부 등) → 0 대체
    # ===============================
    yn_patterns = ['여부']
    yn_cols = [col for col in df.columns if any(p in col for p in yn_patterns)]
    df[yn_cols] = df[yn_cols].fillna(0)
    print(f"Yes/No 컬럼 수: {len(yn_cols)} → 결측 0 대체")

    # ===============================
    # 코드형 변수 (코드로 끝나는 변수) → -1 대체
    # ===============================
    code_cols = [col for col in df.columns if col.endswith('코드')]
    df[code_cols] = df[code_cols].fillna(-1)
    print(f"카테고리 코드 컬럼 수: {len(code_cols)} → 결측 -1 대체")

    # ===============================
    # 남은 결측: 연속형 → 중앙값, 범주형 → '미응답'   (만족감)
    # ===============================
    remain_cols = df.columns[df.isnull().any()]
    for col in remain_cols:
        if df[col].dtype in ['float64', 'int64']:
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna('미응답')
    print(f"남은 결측 처리: {len(remain_cols)} 개")

    # ===============================
    # 최종 확인
    print(f"최종 남은 결측치 수: {df.isnull().sum().sum()}")
    return df

# 적용
df_clean = custom_missing_handler(df)


| 처리 단계           | 내용                    |
| --------------- | --------------------- |
| **고결측 삭제**      | 60% 이상 결측 변수는 통째로 제거  |
| **횟수/시간/권수**    | 활동 없음으로 간주 → 0        |
| **참석/납부/소유 여부** | 불참/미납/미소유로 간주 → 0     |
| **코드 변수**       | 응답 안함으로 간주 → -1       |
| **그 외**         | 연속형: 중앙값 / 범주형: '미응답' |

In [ ]:
## 데이터 분할 & 인코딩

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# X, y
X = df_clean.drop(columns=['기부여부'])
y = df_clean['기부여부']

# 범주형 인코딩: 코드형 변수만 라벨인코딩 (또는 OneHot)
code_cols = [col for col in X.columns if col.endswith('코드')]
for col in code_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 연속형 스케일링 (MLP, CNN용)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression
logit = LogisticRegression(max_iter=1000)
logit.fit(X_train_scaled, y_train)

In [ ]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)


In [ ]:
pip install --upgrade scikit-learn xgboost

In [ ]:
# XGBoost
import xgboost as xgb
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# 샘플 데이터
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# XGBoost sklearn API
xgb_clf = xgb.XGBClassifier(n_estimators=10, random_state=42)
xgb_clf.fit(X_train, y_train)

# 안전 출력
print(xgb_clf)

# 예측 확인
print(xgb_clf.score(X_test, y_test))


In [ ]:
# LightGBM
import lightgbm as lgb
lgb_clf = lgb.LGBMClassifier(n_estimators=100, random_state=42)
lgb_clf.fit(X_train, y_train)

In [ ]:
# CatBoost
from catboost import CatBoostClassifier
cat_clf = CatBoostClassifier(iterations=100, verbose=0, random_state=42)
cat_clf.fit(X_train, y_train)

In [ ]:
pip install torch torchvision torchaudio

In [ ]:
# MLP (PyTorch)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# 간단한 MLP
class MLP(nn.Module):
    def __init__(self, input_dim):
        super(MLP, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.layers(x)

mlp_model = MLP(X_train_scaled.shape[1])
# Optimizer, loss, DataLoader, 학습 루프 작성 필요 (원하면 만들어 줄게)

In [ ]:
# 예: CNN으로 연속형 피처 시퀀스 처리 (형식에 맞게 reshape)
class CNN1D(nn.Module):
    def __init__(self, input_dim):
        super(CNN1D, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear((input_dim - 2) * 16, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        x = x.unsqueeze(1)  # (batch, channel, seq)
        return self.conv(x)

In [ ]:
## shap 해석
import shap

# Tree 모델 예시 (XGBoost)
explainer = shap.TreeExplainer(xgb_clf)
shap_values = explainer.shap_values(X_test)

# 전체 변수 중요도
shap.summary_plot(shap_values, X_test)

# 개별 예측 설명
shap.force_plot(explainer.expected_value, shap_values[0,:], X_test.iloc[0,:])

# 의존성 플롯 (상관관계)
#shap.dependence_plot('소득코드', shap_values, X_test)


In [ ]:
#pip show shap

In [ ]:
#pip install --upgrade shap

In [ ]:
#pip install scikit-learn --upgrade

In [ ]:
print(X_train_scaled.shape)
print(X_test_scaled.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
print(X.shape)
print(y.shape)

In [ ]:
#  타깃이 결측인 행 제거
df_clean = df_clean.dropna(subset=['기부여부'])

# X, y 분리
X = df_clean.drop(columns=['기부여부'])
y = df_clean['기부여부']

print(X.shape, y.shape)  # 반드시 행 개수 같아야 함!

# train-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

In [ ]:
# ---------------------------------------------
# 📌 0) 라이브러리 로드
# ---------------------------------------------
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

import shap

# ---------------------------------------------
# 📌 1) 데이터 준비 (df_clean은 이미 결측 처리된 상태라고 가정)
# ---------------------------------------------

# 타깃 결측 제거
df_clean = df_clean.dropna(subset=['기부여부'])

# X, y 분리
X = df_clean.drop(columns=['기부여부'])
y = df_clean['기부여부']

# 코드형 변수 라벨 인코딩
code_cols = [col for col in X.columns if col.endswith('코드')]
for col in code_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

# ---------------------------------------------
# 📌 2) Train/Test split
# ---------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ---------------------------------------------
# 📌 3) 스케일링 (Logistic, MLP 용)
# ---------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------------------------------------
# 📌 4) 모델 정의 & 학습
# ---------------------------------------------

## Logistic Regression
logit = LogisticRegression(max_iter=1000)
logit.fit(X_train_scaled, y_train)

## Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

## XGBoost
xgb_clf = xgb.XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train, y_train)

## LightGBM
lgb_clf = lgb.LGBMClassifier(n_estimators=100, random_state=42)
lgb_clf.fit(X_train, y_train)

## CatBoost
cat_clf = CatBoostClassifier(iterations=100, verbose=0, random_state=42)
cat_clf.fit(X_train, y_train)

# ---------------------------------------------
# 📌 5) 예측 & 성능 비교
# ---------------------------------------------
results = []

# Logistic
y_pred = logit.predict(X_test_scaled)
y_proba = logit.predict_proba(X_test_scaled)[:, 1]
results.append({
    'Model': 'Logistic Regression',
    'Accuracy': accuracy_score(y_test, y_pred),
    'AUC': roc_auc_score(y_test, y_proba),
    'F1': f1_score(y_test, y_pred)
})

# Random Forest
y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]
results.append({
    'Model': 'Random Forest',
    'Accuracy': accuracy_score(y_test, y_pred),
    'AUC': roc_auc_score(y_test, y_proba),
    'F1': f1_score(y_test, y_pred)
})

# XGBoost
y_pred = xgb_clf.predict(X_test)
y_proba = xgb_clf.predict_proba(X_test)[:, 1]
results.append({
    'Model': 'XGBoost',
    'Accuracy': accuracy_score(y_test, y_pred),
    'AUC': roc_auc_score(y_test, y_proba),
    'F1': f1_score(y_test, y_pred)
})

# LightGBM
y_pred = lgb_clf.predict(X_test)
y_proba = lgb_clf.predict_proba(X_test)[:, 1]
results.append({
    'Model': 'LightGBM',
    'Accuracy': accuracy_score(y_test, y_pred),
    'AUC': roc_auc_score(y_test, y_proba),
    'F1': f1_score(y_test, y_pred)
})

# CatBoost
y_pred = cat_clf.predict(X_test)
y_proba = cat_clf.predict_proba(X_test)[:, 1]
results.append({
    'Model': 'CatBoost',
    'Accuracy': accuracy_score(y_test, y_pred),
    'AUC': roc_auc_score(y_test, y_proba),
    'F1': f1_score(y_test, y_pred)
})

# ---------------------------------------------
# 📌 6) 성능표 출력
# ---------------------------------------------
results_df = pd.DataFrame(results).sort_values(by='AUC', ascending=False)
print(results_df)

# ---------------------------------------------
# 📌 7) SHAP 해석 (XGBoost)
# ---------------------------------------------

# 최신 안전: X_train을 background로 넣기
explainer = shap.Explainer(xgb_clf, X_train)
shap_values = explainer(X_test)

# ✅ Global: 변수 중요도 Summary
shap.summary_plot(shap_values, X_test)

# ✅ Local: 개별 예측
shap.initjs()
shap.force_plot(
    shap_values.base_values[0],   # ⭐️ expected_value X
    shap_values.values[0],        # ⭐️ values
    X_test.iloc[0]                # ⭐️ 샘플
)

# ✅ Dependence: 변수 상관
shap.dependence_plot(
    ind = X_test.columns[0],
    shap_values = shap_values.values,
    features = X_test
)


In [ ]:
# ---------------------------------------------
# 📌 PyTorch 추가 설치 라이브러리
# ---------------------------------------------
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ---------------------------------------------
# 📌 MLP 모델 정의
# ---------------------------------------------
class MLP(nn.Module):
    def __init__(self, input_dim):
        super(MLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

# ---------------------------------------------
# 📌 CNN 모델 정의 (1D)
# ---------------------------------------------
class CNN1D(nn.Module):
    def __init__(self, input_dim):
        super(CNN1D, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear((input_dim - 2) * 16, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        x = x.unsqueeze(1)  # (batch, channel, seq_len)
        return self.conv(x)

# ---------------------------------------------
# 📌 TensorDataset & DataLoader
# ---------------------------------------------
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

train_ds = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

# ---------------------------------------------
# 📌 MLP 학습 루프
# ---------------------------------------------
mlp = MLP(X_train_scaled.shape[1])
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.001)

for epoch in range(5):  # 5 epoch 데모 (실전은 50~100)
    mlp.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = mlp(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

# ---------------------------------------------
# 📌 CNN 학습 루프
# ---------------------------------------------
cnn = CNN1D(X_train_scaled.shape[1])
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(cnn.parameters(), lr=0.001)

for epoch in range(5):
    cnn.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = cnn(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

# ---------------------------------------------
# 📌 MLP 성능 계산
# ---------------------------------------------
mlp.eval()
with torch.no_grad():
    y_proba_mlp = mlp(X_test_tensor).numpy().flatten()
    y_pred_mlp = (y_proba_mlp > 0.5).astype(int)

results.append({
    'Model': 'MLP',
    'Accuracy': accuracy_score(y_test, y_pred_mlp),
    'AUC': roc_auc_score(y_test, y_proba_mlp),
    'F1': f1_score(y_test, y_pred_mlp)
})

# ---------------------------------------------
# 📌 CNN 성능 계산
# ---------------------------------------------
cnn.eval()
with torch.no_grad():
    y_proba_cnn = cnn(X_test_tensor).numpy().flatten()
    y_pred_cnn = (y_proba_cnn > 0.5).astype(int)

results.append({
    'Model': 'CNN',
    'Accuracy': accuracy_score(y_test, y_pred_cnn),
    'AUC': roc_auc_score(y_test, y_proba_cnn),
    'F1': f1_score(y_test, y_pred_cnn)
})

# ---------------------------------------------
# ✅ 전체 결과 다시 출력!
# ---------------------------------------------
results_df = pd.DataFrame(results).sort_values(by='AUC', ascending=False)
print(results_df)


In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, balanced_accuracy_score

# ✅ 결과 저장 리스트
results = []

# 📌 모델 리스트
models = [
    ('Logistic Regression', logit, X_test_scaled),
    ('Random Forest', rf, X_test),
    ('XGBoost', xgb_clf, X_test),
    ('LightGBM', lgb_clf, X_test),
    ('CatBoost', cat_clf, X_test)
]

for name, model, X_test_input in models:
    y_pred = model.predict(X_test_input)
    y_proba = model.predict_proba(X_test_input)[:, 1]

    # Confusion Matrix
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    results.append({
        'Dataset': 'Test',
        'Algorithm': name,
        'N': len(y_test),
        'True Positive': tp,
        'True Negative': tn,
        'False Positive': fp,
        'False Negative': fn,
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'Specificity': tn / (tn + fp),
        'F1-score': f1_score(y_test, y_pred),
        'Accuracy': (tp + tn) / (tp + tn + fp + fn),
        'Balanced Accuracy': balanced_accuracy_score(y_test, y_pred),
        'AUC': roc_auc_score(y_test, y_proba)
    })

# ✅ MLP
mlp.eval()
with torch.no_grad():
    y_proba_mlp = mlp(X_test_tensor).numpy().flatten()
    y_pred_mlp = (y_proba_mlp > 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred_mlp).ravel()
results.append({
    'Dataset': 'Test',
    'Algorithm': 'MLP',
    'N': len(y_test),
    'True Positive': tp,
    'True Negative': tn,
    'False Positive': fp,
    'False Negative': fn,
    'Precision': precision_score(y_test, y_pred_mlp),
    'Recall': recall_score(y_test, y_pred_mlp),
    'Specificity': tn / (tn + fp),
    'F1-score': f1_score(y_test, y_pred_mlp),
    'Accuracy': (tp + tn) / (tp + tn + fp + fn),
    'Balanced Accuracy': balanced_accuracy_score(y_test, y_pred_mlp),
    'AUC': roc_auc_score(y_test, y_proba_mlp)
})

# ✅ CNN
cnn.eval()
with torch.no_grad():
    y_proba_cnn = cnn(X_test_tensor).numpy().flatten()
    y_pred_cnn = (y_proba_cnn > 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred_cnn).ravel()
results.append({
    'Dataset': 'Test',
    'Algorithm': 'CNN',
    'N': len(y_test),
    'True Positive': tp,
    'True Negative': tn,
    'False Positive': fp,
    'False Negative': fn,
    'Precision': precision_score(y_test, y_pred_cnn),
    'Recall': recall_score(y_test, y_pred_cnn),
    'Specificity': tn / (tn + fp),
    'F1-score': f1_score(y_test, y_pred_cnn),
    'Accuracy': (tp + tn) / (tp + tn + fp + fn),
    'Balanced Accuracy': balanced_accuracy_score(y_test, y_pred_cnn),
    'AUC': roc_auc_score(y_test, y_proba_cnn)
})

# 최종 테이블 출력
results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
results_df

In [ ]:
# 정확도, 전체 AUC 최우선 LGBM
# 양성(기부자) 놓치면 안 됨 CNN
# False Positive 최소화 (정밀도) Random Forest

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 성능 지표 Barplot
metrics_to_plot = ['Accuracy', 'Balanced Accuracy', 'F1-score', 'AUC']

for metric in metrics_to_plot:
    plt.figure(figsize=(10, 6))
    sns.barplot(x='Algorithm', y=metric, data=results_df)
    plt.title(f'{metric} by Model')
    plt.ylim(0, 1)
    plt.xticks(rotation=45)
    plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix

# LightGBM confusion matrix 예시
y_pred_lgb = lgb_clf.predict(X_test)
cm = confusion_matrix(y_test, y_pred_lgb)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix (LightGBM)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
# SHAP 값을 계산한 상태라고 가정
# shap_values = explainer(X_test)

# 변수 중요도: 평균 절댓값으로 상위 10개
shap_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Mean(|SHAP|)': np.abs(shap_values.values).mean(axis=0)
}).sort_values(by='Mean(|SHAP|)', ascending=False)

top_10_shap = shap_df.head(10)
top_10_shap

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# 모델 예측 준비
y_pred_logit = logit.predict(X_test_scaled)
y_pred_rf = rf.predict(X_test)
y_pred_xgb = xgb_clf.predict(X_test)
y_pred_lgb = lgb_clf.predict(X_test)
y_pred_cat = cat_clf.predict(X_test)

mlp.eval()
with torch.no_grad():
    y_proba_mlp = mlp(X_test_tensor).numpy().flatten()
    y_pred_mlp = (y_proba_mlp > 0.5).astype(int)

cnn.eval()
with torch.no_grad():
    y_proba_cnn = cnn(X_test_tensor).numpy().flatten()
    y_pred_cnn = (y_proba_cnn > 0.5).astype(int)

# 모든 모델 혼동 행렬
model_preds = {
    'Logistic Regression': y_pred_logit,
    'Random Forest': y_pred_rf,
    'XGBoost': y_pred_xgb,
    'LightGBM': y_pred_lgb,
    'CatBoost': y_pred_cat,
    'MLP': y_pred_mlp,
    'CNN': y_pred_cnn
}

for name, y_pred in model_preds.items():
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix ({name})')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()


In [ ]:
import shap

# LightGBM SHAP
explainer_lgb = shap.Explainer(lgb_clf, X_train)
shap_values_lgb = explainer_lgb(X_test)

shap.summary_plot(shap_values_lgb, X_test)

# Random Forest SHAP
explainer_rf = shap.Explainer(rf, X_train)
shap_values_rf = explainer_rf(X_test)

shap.summary_plot(shap_values_rf, X_test)

# CNN SHAP (Kernel 예시)
import numpy as np
explainer_cnn = shap.KernelExplainer(
    lambda x: cnn(torch.tensor(x, dtype=torch.float32)).detach().numpy(),
    X_train_scaled[:100]  # Background sample
)

shap_values_cnn = explainer_cnn.shap_values(X_test_scaled[:50])  # 계산량 크므로 일부 샘플
shap.summary_plot(shap_values_cnn, X_test.iloc[:50])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 상관계수 테이블
corr_matrix = df_clean.corr(numeric_only=True)  # 숫자형 변수만 자동 처리
print(corr_matrix)

# '기부여부' 와 상관계수만 정렬 출력
corr_target = corr_matrix['기부여부'].sort_values(ascending=False)
print("\n[기부여부]와의 상관계수")
print(corr_target)